# Spark 01: Introducción a PySpark, DataFrames y SparkSQL
**Universidad del Valle de Guatemala — Data Science**

Esta libreta introduce los conceptos fundamentales de la arquitectura de Apache Spark, la manipulación de datos semiestructurados (JSON) mediante la API de DataFrames y la ejecución de consultas optimizadas con SparkSQL.

### 1. Verificación del Entorno de Ejecución (Java y PySpark)
Apache Spark está construido sobre Scala y se ejecuta sobre la **Java Virtual Machine (JVM)**. Para que PySpark pueda comunicarse con el motor subyacente a través de `Py4J`, el contenedor debe tener configurado `JAVA_HOME` apuntando a OpenJDK 17 y PySpark 3.5+ instalado.

In [3]:
import os
import pyspark
from pyspark.sql import SparkSession

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("PySpark:", pyspark.__version__)

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
PySpark: 3.5.1


### 2. Inicialización de la SparkSession
La `SparkSession` es el punto de entrada unificado para interactuar con todas las funcionalidades de Spark.
* `.appName("NotebookPySpark")`: Nombre descriptivo para identificar el trabajo en la interfaz web de Spark.
* `.master("local[*]")`: Indica que Spark correrá en modo local utilizando todos los núcleos de CPU disponibles (`*`) dentro del contenedor.
* `.config("spark.driver.host", "127.0.0.1")`: Configuración requerida en entornos Docker para enlazar el Driver a la interfaz de red local.
* `.getOrCreate()`: Obtiene la sesión activa existente o crea una nueva si aún no ha sido iniciada.

In [4]:
spark = (
    SparkSession.builder
    .appName("NotebookPySpark")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

print("Spark:", spark.version)
print("Java:", spark.sparkContext._jvm.java.lang.System.getProperty("java.version"))

Spark: 3.5.1
Java: 17.0.20.1


### 3. Definición de la Carpeta de Trabajo
Definimos la ruta hacia el volumen montado `/opt/app/working_dir/`, donde residen los datos de tweets y partidos de baloncesto.

In [5]:
mydir = "/opt/app/working_dir/"

### 4. Carga de Datos Semiestructurados (JSON) e Inferencia de Esquema
A diferencia de los archivos tabulares simples, los archivos JSON suelen contener estructuras anidadas y jerárquicas.
* `spark.read.json(...)`: Spark analiza el archivo de tweets e infiere automáticamente el esquema.
* `printSchema()`: Imprime la estructura jerárquica en forma de árbol. Nota cómo existen tipos primitivos (`string`, `long`) y tipos complejos como `struct` (ej. `user`) y `array` (ej. `hashtags`).

In [6]:
trump = spark.read.json(mydir + 'trump_tweets/donald_data.json')
trump.printSchema()

root
 |-- contributors: string (nullable = true)
 |-- coordinates: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- display_text_range: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- entities: struct (nullable = true)
 |    |-- hashtags: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- indices: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |    |    |-- text: string (nullable = true)
 |    |-- media: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- display_url: string (nullable = true)
 |    |    |    |-- expanded_url: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- id_str: string (nullable = true)
 |    |    |    |-- indices: array (nullable = true)
 |    |    |    |    |-- element: long (containsNull = true)
 |    |    |    |-- media_url: string (nulla

26/09/21 15:55:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### 5. Verificación del Tipo de Objeto
Comprobamos que el objeto resultante es un `pyspark.sql.dataframe.DataFrame`. A diferencia de Pandas, este ***DataFrame es una colección inmutable y particionada de registros distribuidos.***

In [8]:
type(trump)

pyspark.sql.dataframe.DataFrame

### 6. Visualización de Datos con `.show()`
`.show()` es una **Acción** que desencadena la ejecución del grafo lógico de Spark.
* Por defecto muestra las primeras 20 filas y trunca las cadenas largas a 20 caracteres.
* Al pasar `truncate=False`, podemos visualizar el contenido íntegro de cada campo de texto sin recortes.

In [9]:
trump.show()

+------------+-----------+--------------------+------------------+--------------------+--------------------+--------------+---------+--------------------+----+-------------------+-------------------+-----------------------+---------------------+-------------------------+-------------------+-----------------------+---------------+----+-----+------------------+--------------------+-------------------+--------------------+-----------------------+-------------+---------+--------------------+---------+--------------------+
|contributors|coordinates|          created_at|display_text_range|            entities|   extended_entities|favorite_count|favorited|           full_text| geo|                 id|             id_str|in_reply_to_screen_name|in_reply_to_status_id|in_reply_to_status_id_str|in_reply_to_user_id|in_reply_to_user_id_str|is_quote_status|lang|place|possibly_sensitive|       quoted_status|   quoted_status_id|quoted_status_id_str|quoted_status_permalink|retweet_count|retweeted|     

In [10]:
trump.show(truncate=False)

+------------+-----------+------------------------------+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### 7. Inspección de Campos Anidados en Formato Vertical
Cuando una columna contiene un objeto complejo (`struct` o `array`), como `extended_entities`, una visualización tabular horizontal resulta difícil de leer.
* `show(1, False, True)`:
  - `1`: Solo la primera fila.
  - `False`: Sin truncar texto.
  - `True` (`vertical=True`): Muestra cada atributo en formato vertical (clave: valor).

In [11]:
# numRows, truncate y vertical.
trump.select("extended_entities").show(1, False, True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


### 8. Selección de Columnas y Acceso con Notación de Punto
La función `.select()` es una **Transformación** (evaluación perezosa). En Spark, podemos acceder directamente a campos anidados dentro de un `struct` mediante notación de punto (ej. `'user.screen_name'`) sin necesidad de desanidar manualmente el JSON.

In [12]:
trump_clean = trump.select(
  'user.screen_name',
  'created_at',
  'full_text',
  'retweet_count',
  'favorite_count')

trump_clean.show()

+---------------+--------------------+--------------------+-------------+--------------+
|    screen_name|          created_at|           full_text|retweet_count|favorite_count|
+---------------+--------------------+--------------------+-------------+--------------+
|realDonaldTrump|Tue Sep 11 20:16:...|The safety of Ame...|         7270|         27341|
|realDonaldTrump|Tue Sep 11 16:48:...|Small Business Op...|         8520|         31085|
|realDonaldTrump|Tue Sep 11 15:32:...|#NeverForget #Sep...|         9641|         33968|
|realDonaldTrump|Tue Sep 11 12:58:...|17 years since Se...|        17952|         73372|
|realDonaldTrump|Tue Sep 11 12:24:...|Departing Washing...|        12541|         52227|
|realDonaldTrump|Tue Sep 11 11:59:...|Rudy Giuliani did...|        19674|         90609|
|realDonaldTrump|Tue Sep 11 11:41:...|“ERIC Holder coul...|        10760|         43470|
|realDonaldTrump|Tue Sep 11 11:19:...|New Strzok-Page t...|        18090|         64190|
|realDonaldTrump|Tue 

### 9. Conteo Distribuido de Filas (`.count()`)
`.count()` es una **Acción**. Spark lanza un Job distribuido en el clúster para contar el número de registros en cada partición y devuelve el total consolidado al nodo Driver.

In [13]:
trump_clean.count()

2865

### 10. Transformación y Limpieza con `pyspark.sql.functions`
El módulo `functions` (importado convencionalmente como `f`) proporciona operaciones nativas altamente optimizadas en la JVM de Spark.
* `f.encode('full_text', 'ascii')`: Normaliza el texto a caracteres ASCII (removiendo emojis y caracteres incompatibles).
* `.cast('string')`: Convierte el tipo binario resultante nuevamente a texto plano.
* `.alias('text')`: Renombra la columna transformada.

In [14]:
from pyspark.sql import functions as f

trump_clean = trump.select(
  'user.screen_name',
  'created_at',
  f.encode('full_text', 'ascii').cast('string').alias('text'),
  'retweet_count',
  'favorite_count')

trump_clean.show()

+---------------+--------------------+--------------------+-------------+--------------+
|    screen_name|          created_at|                text|retweet_count|favorite_count|
+---------------+--------------------+--------------------+-------------+--------------+
|realDonaldTrump|Tue Sep 11 20:16:...|The safety of Ame...|         7270|         27341|
|realDonaldTrump|Tue Sep 11 16:48:...|Small Business Op...|         8520|         31085|
|realDonaldTrump|Tue Sep 11 15:32:...|#NeverForget #Sep...|         9641|         33968|
|realDonaldTrump|Tue Sep 11 12:58:...|17 years since Se...|        17952|         73372|
|realDonaldTrump|Tue Sep 11 12:24:...|Departing Washing...|        12541|         52227|
|realDonaldTrump|Tue Sep 11 11:59:...|Rudy Giuliani did...|        19674|         90609|
|realDonaldTrump|Tue Sep 11 11:41:...|?ERIC Holder coul...|        10760|         43470|
|realDonaldTrump|Tue Sep 11 11:19:...|New Strzok-Page t...|        18090|         64190|
|realDonaldTrump|Tue 

### 11. Escritura Distribuida a Disco (Múltiples Particiones)
Guardamos el DataFrame en disco en formato CSV utilizando un delimitador de barra vertical (`|`) para evitar conflictos con comas dentro del texto de los tweets.
* **Comportamiento distribuido:** Spark creará un directorio `output/` que contiene múltiples archivos `part-*.csv`. Cada archivo es generado en paralelo por una partición distinta.

In [15]:
trump_clean.write \
    .format("csv") \
    .option("header", "true") \
    .option("delimiter", "|") \
    .mode("overwrite") \
    .save("output/")

### 12. Consolidación de Particiones con `coalesce(1)`
* `.coalesce(1)`: Reduce el número de particiones del DataFrame a exactamente 1 sin disparar un Shuffle completo por la red.
* **Resultado:** La carpeta `output2/` contendrá un único archivo CSV consolidado.
* **Advertencia técnica:** En Big Data masivo, `coalesce(1)` puede colapsar la memoria de un único nodo ejecutor. Úsalo solo para resultados agregados o de tamaño controlado.

In [16]:
trump_clean.coalesce(1).write \
    .format("csv") \
    .option("header", "true") \
    .option("delimiter", "|") \
    .mode("overwrite") \
    .save("output2/")

In [17]:
trump_clean.show(n=3, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                   
 created_at     | Tue Sep 11 20:16:49 +0000 2018                                                                                                                                                                                    
 text           | The safety of American people is my absolute highest priority. Heed the directions of your State and Local Officials. Please be prepared, be careful and be SAFE! https://t.co/YP7ssITwW9 https://t.co/LZIUCgdPTH 
 retweet_count  | 7270                                                              

### 13. Ordenamiento y Filtrado de los Tweets más Populares
* `.orderBy(f.desc("retweet_count"))`: Ordena todo el conjunto distribuido de forma descendente por la columna `retweet_count`.
* `.limit(3)`: Extrae únicamente los 3 primeros registros (los más virales).

In [18]:
top3_rt = trump_clean.orderBy(f.desc("retweet_count")).limit(3)
top3_rt.show(n=3, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                                                                                              
 created_at     | Sun Nov 12 00:48:01 +0000 2017                                                                                                                                                                                                                                                               
 text           | Why would Kim Jong-un insult me by calling me "old," when I would NEVE

### 14. Consultas con SparkSQL y Vistas Temporales
Spark permite consultar DataFrames utilizando sintaxis estándar de SQL gracias al optimizador Catalyst.
* `.createOrReplaceTempView("tweets")`: Registra el DataFrame en el catálogo de Spark como una tabla virtual temporal llamada `tweets`.
* `spark.sql(query)`: Ejecuta la consulta SQL directamente y retorna un nuevo DataFrame de Spark optimizado.

In [19]:
trump_clean.createOrReplaceTempView("tweets")

# Consulta SQL equivalente para obtener los 3 tweets más retuiteados
query = """
    SELECT *
    FROM tweets
    ORDER BY retweet_count DESC
    LIMIT 3
"""

top_3_tweets = spark.sql(query)
top_3_tweets.show(n=3, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 screen_name    | realDonaldTrump                                                                                                                                                                                                                                                                              
 created_at     | Sun Nov 12 00:48:01 +0000 2017                                                                                                                                                                                                                                                               
 text           | Why would Kim Jong-un insult me by calling me "old," when I would NEVE